Varshini Narayanan

This code scrapes data on the largest publically traded companies by market cap. Foreign Domiciled and repeat corporations are manually removed.

51 lines of code

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# request page
url = "https://finance.yahoo.com/research-hub/screener/largest_market_cap/"

request_headers = {
    "User-Agent": "For the purpose of a class project varshi@uchicago.edu "
}

response = requests.get(url, headers=request_headers)
soup = BeautifulSoup(response.text, "html.parser")

# find table
table = soup.find("table")

# find the correct headers
column_headers = [
    th.text.strip()
    for th in table.find("thead").find_all("th")
]

# align to headers
rows = table.find("tbody").find_all("tr")

data = []
for row in rows:
    cells = [td.text.strip() for td in row.find_all("td")]
    data.append(dict(zip(column_headers, cells)))

# made dataframe
df = pd.DataFrame(data)

# Clean ticker symbols
df["Symbol"] = df["Symbol"].str.split().str[-1]


df


,,Symbol,Name,1D Chart,Price (Intraday),Change,Change %,Volume,Avg Vol (3M),Market Cap,P/E Ratio (TTM),52 Week Range,Region,Follow
0,1,AZULD,Azul S.A.,,"1,066.67",0.00,0.00%,1,19,"1,451.912T",--,"670.00 39,750.00",US,
1,2,NVDA,NVIDIA Corporation,,172.05,-2.14,-1.23%,177.723M,181.589M,4.189T,44.64,86.62 212.19,US,
2,3,AAPL,Apple Inc.,,275.91,-0.58,-0.21%,49.149M,47.944M,4.055T,34.18,169.21 288.62,US,
3,4,GOOGL,Alphabet Inc.,,331.54,-1.50,-0.45%,79.776M,36.103M,4.016T,33.52,140.53 349.00,US,
4,5,GOOG,Alphabet Inc.,,331.33,-2.01,-0.60%,47.323M,23.254M,4T,33.52,142.66 350.15,US,
5,6,MSFT,Microsoft Corporation,,393.74,-20.45,-4.94%,55.083M,28.583M,2.926T,25.73,344.79 555.45,US,
6,7,AMZN,"Amazon.com, Inc.",,222.90,-10.09,-4.33%,68.885M,40.68M,2.383T,33.72,161.38 258.60,US,
7,8,TSM,Taiwan Semiconductor Manufacturing Company Lim...,,330.73,+4.99,+1.53%,13.769M,12.674M,1.715T,27.17,134.25 351.33,US,
8,9,META,"Meta Platforms, Inc.",,670.33,+1.34,+0.20%,14.172M,17.617M,1.696T,29.45,479.80 796.25,US,
9,10,TSLA,"Tesla, Inc.",,397.36,-8.65,-2.13%,65.297M,72.701M,1.491T,390.70,214.25 498.83,US,


In [34]:
#remove foreign domicilced companies

df["Change"].dtype
df = df[df["Change"] != 0.0]

#remove tencent seperately
df = df[~df["Symbol"].isin(["TCTZF", "TCEHY"])]

In [ ]:
#convert market cap values from str (13T) into numerical value

def parse_market_cap(x):
    if pd.isna(x):
        return None
    x = x.replace(",", "")
    if x.endswith("T"):
        return float(x[:-1]) * 1e12
    if x.endswith("B"):
        return float(x[:-1]) * 1e9
    if x.endswith("M"):
        return float(x[:-1]) * 1e6
    return float(x)

df["Market Cap Num"] = df["Market Cap"].apply(parse_market_cap)


/var/folders/vy/c51m8q751_72tx9f0b1r4blc0000gn/T/ipykernel_49721/3386891294.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Market Cap Num"] = df["Market Cap"].apply(parse_market_cap)


In [35]:
#keep only largest marketcap listing for companies that appear multiple times
df = (
    df.sort_values("Market Cap Num", ascending=False)
      .drop_duplicates(subset="Name", keep="first")
      .reset_index(drop=True)
)

df

,,Symbol,Name,1D Chart,Price (Intraday),Change,Change %,Volume,Avg Vol (3M),Market Cap,P/E Ratio (TTM),52 Week Range,Region,Follow,Market Cap Num
0,2,NVDA,NVIDIA Corporation,,172.05,-2.14,-1.23%,177.723M,181.589M,4.189T,44.64,86.62 212.19,US,,4.189000e+12
1,3,AAPL,Apple Inc.,,275.91,-0.58,-0.21%,49.149M,47.944M,4.055T,34.18,169.21 288.62,US,,4.055000e+12
2,4,GOOGL,Alphabet Inc.,,331.54,-1.50,-0.45%,79.776M,36.103M,4.016T,33.52,140.53 349.00,US,,4.016000e+12
3,6,MSFT,Microsoft Corporation,,393.74,-20.45,-4.94%,55.083M,28.583M,2.926T,25.73,344.79 555.45,US,,2.926000e+12
4,7,AMZN,"Amazon.com, Inc.",,222.90,-10.09,-4.33%,68.885M,40.68M,2.383T,33.72,161.38 258.60,US,,2.383000e+12
5,8,TSM,Taiwan Semiconductor Manufacturing Company Lim...,,330.73,4.99,+1.53%,13.769M,12.674M,1.715T,27.17,134.25 351.33,US,,1.715000e+12
6,9,META,"Meta Platforms, Inc.",,670.33,1.34,+0.20%,14.172M,17.617M,1.696T,29.45,479.80 796.25,US,,1.696000e+12
7,10,TSLA,"Tesla, Inc.",,397.36,-8.65,-2.13%,65.297M,72.701M,1.491T,390.70,214.25 498.83,US,,1.491000e+12
8,11,AVGO,Broadcom Inc.,,310.51,2.46,+0.80%,37.996M,30.735M,1.472T,67.16,138.10 414.61,US,,1.472000e+12
9,12,BRK-A,Berkshire Hathaway Inc.,,"756,943.00",-947.00,-0.12%,248,785,1.089T,15.78,"685,150.00 812,855.00",US,,1.089000e+12


In [ ]:
#save as csv
df.to_csv("firm_list.csv", index=False)